# Week 1 Quiz — 16 Practice Questions

**Topics:** Spark architecture, deployment modes, DataFrame basics, column operations,
filtering/nulls, Spark SQL (DDL/DML), reading and writing data.

**Instructions:** Read each question, choose your answer, then run the code cell below to reveal the correct answer and explanation. Aim for 13/16 (80%) before moving to Week 2.

---

**Q1.** Which Spark deployment mode requires ALL executors to run on a single worker node?

A. Cluster mode  
B. Local mode  
C. Client mode  
D. Standard mode

In [ ]:
# ANSWER: B
# Local mode runs the driver AND all executors inside a single JVM on one machine.
# Client mode: driver on submitting machine, executors on cluster nodes.
# Cluster mode: driver runs on a cluster node (most common in production).
print("Q1 → B  (Local mode — single machine, one JVM)")

**Q2.** Which code block replaces the `division` column in `storesDF` with a new column named `state`, AND renames `mName` to `managerName`?

A.
```python
storesDF = (storesDF.withColumn("state", col("division"))
                    .drop("division")
                    .withColumnRenamed("mName", "managerName"))
```
B.
```python
storesDF = (storesDF.withColumnRenamed("state", "division")
                    .withColumnRenamed("mName", "managerName"))
```
C.
```python
storesDF = (storesDF.withColumn(col("division"), state)
                    .withColumnRenamed("managerName", "mName"))
```
D.
```python
storesDF = (storesDF.drop("division")
                    .withColumnRenamed("state", lit("default_state"))
                    .withColumnRenamed(columns={"mName": "managerName"}))
```

In [ ]:
# ANSWER: A
# Step 1: withColumn("state", col("division")) — creates new column "state" from "division"
# Step 2: drop("division") — removes old "division" column
# Step 3: withColumnRenamed("mName", "managerName") — renames mName
# B is wrong: withColumnRenamed("state", "division") renames state TO division (backwards)
# C: first arg to withColumn must be a string, not col(); also backwards rename
# D: withColumnRenamed does not accept a dict or lit() as arguments
from pyspark.sql.functions import col
df = spark.createDataFrame([(1, "North", "Mgr A")], ["id", "division", "mName"])
result = (df.withColumn("state", col("division")).drop("division").withColumnRenamed("mName", "managerName"))
result.show()
print("Q2 → A")

**Q3.** Given `employeeDF` with columns `name`, `department`, `salary`, `age`. Which snippet correctly sorts by multiple columns descending, prints schema, and converts to a list of row dicts?

A.
```python
result = employeeDF.orderBy(desc("salary"), desc("age"))
employeeDF.printSchema()
row_list = result.toPandas().values.tolist()
```
B.
```python
result = employeeDF.sort("salary", "age", ascending=[False, False])
employeeDF.schema()
row_list = result.toList()
```
C.
```python
result = (employeeDF.orderBy(col("salary").desc(), col("age").desc()).collect())
employeeDF.printSchema()
row_list = [row.asDict() for row in result]
```
D.
```python
result = (employeeDF.sort(["salary", "age"], descending=True).show())
employeeDF.describe()
row_list = list(result)
```

In [ ]:
# ANSWER: C
# C correctly: orderBy(col().desc(), col().desc()) + collect() + asDict()
# A: toPandas().values.tolist() returns list of lists, not row dicts. Also toPandas collects all data.
# B: schema() is not a method (printSchema() is); toList() doesn't exist in PySpark
# D: sort() has no 'descending' parameter; show() returns None so row_list = list(None) fails
from pyspark.sql.functions import col
df = spark.createDataFrame([("Alice","Eng",95000,30),("Bob","Mkt",72000,25)],["name","dept","salary","age"])
result = (df.orderBy(col("salary").desc(), col("age").desc()).collect())
df.printSchema()
row_list = [row.asDict() for row in result]
print(row_list)
print("Q3 → C")

**Q4.** Which code fragment returns a new DataFrame from `storesDF` that excludes ALL rows containing at least one missing value in ANY column?

A. `storesDF.na.drop("all")`  
B. `storesDF.na.drop(subset="sqft")`  
C. `storesDF.na.drop()`  
D. `storesDF.dropna("all")`

In [ ]:
# ANSWER: C
# na.drop() with no arguments drops rows where ANY column is null (default how='any')
# A: na.drop('all') drops rows only when ALL columns are null — too permissive
# B: na.drop(subset='sqft') only checks the sqft column
# D: dropna('all') is equivalent to A (drops only all-null rows), not any-null rows
df = spark.createDataFrame([(1, None, 3),(2, 2, None),(3, 3, 3),(None, None, None)],["a","b","c"])
print("na.drop() result (removes any null row):", df.na.drop().count())   # 1 row left
print("na.drop('all') result (removes all-null row):", df.na.drop('all').count())  # 3 rows left
print("Q4 → C")

**Q5.** A data engineer needs to write `df` to Parquet, partitioned by `country`, overwriting any existing data. Which code is correct?

A. `df.write.mode("append").partitionBy("country").parquet("/data/output")`  
B. `df.write.partitionBy("country").parquet("/data/output")`  
C. `df.write.mode("overwrite").parquet("/data/output")`  
D. `df.write.mode("overwrite").partitionBy("country").parquet("/data/output")`

In [ ]:
# ANSWER: D
# Requires both mode('overwrite') AND partitionBy('country') AND parquet(path)
# A: append mode doesn't overwrite
# B: no mode means default (error if exists)
# C: no partitionBy — data won't be partitioned by country
import tempfile
path = tempfile.mkdtemp()
df = spark.createDataFrame([("Alice","RO"),("Bob","UK")],["name","country"])
df.write.mode("overwrite").partitionBy("country").parquet(path)
print("Written. Partitions:", spark.read.parquet(path).rdd.getNumPartitions())
print("Q5 → D")

**Q6.** Which SQL statement correctly demonstrates a DDL (Data Definition Language) operation to CREATE a table?

A. `CREATE TABLE employees (id INT, name STRING);`  
B. `INSERT INTO employees (id, name) VALUES (1, 'Alice');`  
C. `ALTER TABLE employees ADD COLUMN salary DECIMAL(10,2);`  
D. `DROP TABLE employees;`

In [ ]:
# ANSWER: A
# DDL = Data Definition Language: CREATE TABLE, ALTER TABLE, DROP TABLE (schema structure)
# B is DML (Data Manipulation Language) — it inserts rows of data
# C is also DDL but modifies an existing table, doesn't CREATE it
# D is DDL but removes the table, doesn't create it
# The question specifically asks for CREATE, so A is the answer
print("Q6 → A  (CREATE TABLE is DDL — defines schema structure)")

**Q7.** A junior data engineer needs to create a Spark-managed table where Spark controls both data and metadata, stored in DBFS. Which command is correct?

A. `CREATE TABLE my_table (id STRING, value STRING) USING parquet LOCATION 'dbfs:/user/hive/warehouse/my_table'`  
B. `CREATE TABLE my_table (id STRING, value STRING) USING parquet LOCATION 'dbfs:/user/hive/warehouse'`  
C. `CREATE MANAGED TABLE my_table (id STRING, value STRING);`  
D. `CREATE EXTERNAL my_table (id STRING, value STRING) USING DBFS;`  
E. `CREATE TABLE my_table (id STRING, value STRING);`

In [ ]:
# ANSWER: E
# A managed table omits the LOCATION clause — Spark picks the default DBFS warehouse path.
# A and B: providing LOCATION makes it an EXTERNAL table (Spark only manages metadata, not data)
# C: 'CREATE MANAGED TABLE' is not valid Spark SQL syntax
# D: 'CREATE EXTERNAL ...' syntax is wrong and 'USING DBFS' is not a valid format
spark.sql("CREATE DATABASE IF NOT EXISTS quiz_db")
spark.sql("USE quiz_db")
spark.sql("DROP TABLE IF EXISTS my_table_managed")
spark.sql("CREATE TABLE my_table_managed (id STRING, value STRING)")
print("Table type:", spark.catalog.getTable("quiz_db.my_table_managed").tableType)
print("Q7 → E")

**Q8.** A data engineer wants to create a relational object from multiple tables, accessible by other engineers across sessions, WITHOUT copying physical data. Which object should be created?

A. View  
B. Temporary view  
C. Delta Table  
D. Database  
E. Spark SQL Table

In [ ]:
# ANSWER: A
# A regular View: persisted in metastore, no physical data copy, accessible cross-session
# B: Temporary view is session-scoped — gone when session ends
# C: Delta Table stores physical data — incurs storage costs
# D: Database is a namespace/container, not a data entity
# E: A Spark SQL Table (managed/external) stores physical data
print("Q8 → A  (View = virtual, no physical copy, persistent across sessions)")

**Q9.** A data engineer wants to create a data entity from multiple tables that displays combined data AND saves to a physical location. Which object should be created?

A. View  
B. Function  
C. Database  
D. Table

In [ ]:
# ANSWER: D
# A Table (Delta or Parquet) stores data physically and can be created from multiple sources via CTAS
# A: View doesn't store physical data
# B: Function is reusable logic, not a data entity
# C: Database is a namespace/container
print("Q9 → D  (Table = physical storage + can combine multiple source tables)")

**Q10.** How do you access a Spark SQL table named `sales` in PySpark to work with it as a DataFrame?

A. `SELECT * FROM sales`  
B. There is no way to share data between PySpark and SQL  
C. `spark.sql("sales")`  
D. `spark.data.table("sales")`  
E. `spark.table("sales")`

In [ ]:
# ANSWER: E
# spark.table("table_name") returns a PySpark DataFrame from a registered table
# A: SELECT ... is SQL syntax, not a PySpark command on its own
# B: False — Spark fully supports SQL/PySpark interoperability via shared metastore
# C: spark.sql() requires a full SQL query string, not just a table name
# D: spark.data.table() doesn't exist in the PySpark API
from pyspark.sql.functions import col
df = spark.createDataFrame([(1,100),(2,200)],["id","amount"])
df.createOrReplaceTempView("sales")
result = spark.table("sales")  # same as above DataFrame
result.show()
print("Q10 → E")

**Q11.** Which SQL keyword appends new rows to an existing Delta table?

A. UPDATE  
B. COPY  
C. INSERT INTO  
D. DELETE  
E. UNION

In [ ]:
# ANSWER: C
# INSERT INTO appends new rows without removing existing data
# A: UPDATE modifies existing rows
# B: COPY INTO is a separate Databricks command for loading files; 'COPY' alone is not valid
# D: DELETE removes rows
# E: UNION combines result sets in a query; it doesn't write data
print("Q11 → C  (INSERT INTO appends rows to an existing table)")

**Q12.** A data engineer has a new record: `id STRING = 'a1'`, `rank INTEGER = 6`, `rating FLOAT = 9.4`. Which SQL command appends it to `my_table`?

A. `INSERT INTO my_table VALUES ('a1', 6, 9.4)`  
B. `INSERT VALUES ('a1', 6, 9.4) INTO my_table`  
C. `UPDATE my_table VALUES ('a1', 6, 9.4)`  
D. `UPDATE VALUES ('a1', 6, 9.4) my_table`

In [ ]:
# ANSWER: A
# Standard SQL: INSERT INTO table VALUES (val1, val2, ...)
# B: Wrong word order — table name must follow INSERT INTO immediately
# C: UPDATE does not use VALUES clause like this — UPDATE requires SET col = val
# D: Completely invalid SQL syntax
print("Q12 → A  (INSERT INTO table VALUES (v1, v2, ...) is correct SQL)")

**Q13.** A data engineer creates a table from a CSV file:
```sql
CREATE TABLE new_table
____
OPTIONS (header = "true", delimiter = "|")
LOCATION "/path/to/csv"
```
Which line fills the blank to specify the CSV format?

A. `FROM "/path/to/csv"`  
B. `USING CSV`  
C. `FROM CSV`  
D. `USING DELTA`

In [ ]:
# ANSWER: B
# USING specifies the data source format in Spark SQL CREATE TABLE
# 'USING CSV' tells Spark to use the CSV reader with the given OPTIONS
# A and C: FROM is used in SELECT queries, not CREATE TABLE
# D: USING DELTA creates a Delta table — Delta doesn't use header/delimiter OPTIONS
print("Q13 → B  (USING CSV — USING clause sets the data source format)")

**Q14.** Which code returns a new DataFrame with a new column `sqft100` that is 1/100th of column `sqft` in `storesDF`?

A. `storesDF.withColumn("sqft100", col("sqft") * 100)`  
B. `storesDF.withColumn("sqft100", sqft / 100)`  
C. `storesDF.withColumn(col("sqft100"), col("sqft") / 100)`  
D. `storesDF.withColumn("sqft100", col("sqft") / 100)`  
E. `storesDF.newColumn("sqft100", sqft / 100)`

In [ ]:
# ANSWER: D
# withColumn(string_name, Column_expression) — both types correct
# A: multiplies by 100, not divides (gives 100× sqft, not 1/100)
# B: bare 'sqft' is not a valid Python identifier; needs col("sqft")
# C: first arg must be a plain string, not col("sqft100") — raises error
# E: newColumn() doesn't exist in the Spark DataFrame API
from pyspark.sql.functions import col
storesDF = spark.createDataFrame([(1, 1000), (2, 2000)], ["id", "sqft"])
storesDF.withColumn("sqft100", col("sqft") / 100).show()
print("Q14 → D")

**Q15.** A data engineer has a Python variable `table_name` they want to use in a Spark SQL query. Which approach correctly runs the query using the variable?

A. `spark.delta.sql(f"SELECT customer_id, spend FROM {table_name}")`  
B. `spark.sql(f"SELECT customer_id, spend FROM {table_name}")`  
C. `spark.table(f"SELECT customer_id, spend FROM {table_name}")`  
D. `dbutils.sql(f"SELECT customer_id, spend FROM {table_name}")`

In [ ]:
# ANSWER: B
# spark.sql() accepts a SQL string; f-strings inject Python variables
# A: spark.delta.sql does not exist
# C: spark.table() takes a table name string only, not a full SQL query
# D: dbutils has no .sql() method
table_name = "sales"
result = spark.sql(f"SELECT * FROM {table_name}")
result.show()
print("Q15 → B")

**Q16.** Why is creating an external table from Parquet preferred over CSV for performance?

A. Parquet files can be partitioned (CSV cannot)  
B. CREATE TABLE AS SELECT cannot be used with CSV files  
C. Parquet files have a well-defined schema embedded in the file  
D. Parquet files automatically become Delta tables  
E. Parquet files are always smaller than CSV files

In [ ]:
# ANSWER: C
# Parquet embeds column names, types, and nullability in file metadata → schema inference works
# CSV has no schema → must manually define all columns in CREATE TABLE
# A: CSV CAN be partitioned with Hive-style directories
# B: CTAS can be used with both formats
# D: Creating an external table on Parquet does NOT make it a Delta table
# E: Not always true; compressed CSV can sometimes be smaller depending on data
print("Q16 → C  (Parquet stores schema metadata — enables reliable schema inference)")

---
## Quiz Score

| Q | Topic | Your answer | Correct |
|---|---|---|---|
| 1 | Deployment modes | | B |
| 2 | withColumn / drop / rename | | A |
| 3 | sort / printSchema / collect | | C |
| 4 | na.drop null handling | | C |
| 5 | Write partitionBy overwrite | | D |
| 6 | DDL CREATE TABLE | | A |
| 7 | Managed table creation | | E |
| 8 | View (no physical data) | | A |
| 9 | Table (physical data) | | D |
| 10 | spark.table() in PySpark | | E |
| 11 | INSERT INTO append | | C |
| 12 | INSERT INTO VALUES syntax | | A |
| 13 | USING CSV clause | | B |
| 14 | withColumn / 100 | | D |
| 15 | spark.sql with Python variable | | B |
| 16 | Parquet schema benefit | | C |

**Target: 13/16 (80%) before moving to Week 2.**